In [1]:
from ast import literal_eval  # as tuple literal_eval -> convert string representation of list to actual list
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display


In [2]:
# --- par‑plant ---------------------------------------------------------------
path_box = Path("/home/loai/Documents/code/RSMLExtraction/Results/Reconstruction/measure_per_plant.csv")
df_plant_wid = pd.read_csv(path_box)
df_plant_wid['root_ids'] = df_plant_wid['root_ids'].apply(literal_eval)
df_plant_wid

,model,split,box,metric,time,root_ids,source,value
0,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",Prediction,1.000000
1,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",expertized,1.000000
2,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",before_expertized,1.000000
3,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(36, 47, 45)",Prediction,1.000000
4,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(36, 47, 45)",expertized,1.000000
...,...,...,...,...,...,...,...,...
209491,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(10, 14, 8)",expertized,125.584577
209492,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(10, 14, 8)",before_expertized,20.126268
209493,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(17, 23, 13)",Prediction,4.000000
209494,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(17, 23, 13)",expertized,119.834592


In [3]:
df_grouped = (
    df_plant_wid
    .set_index(['model', 'split', 'box', 'metric', 'root_ids', 'source', 'time'])
    .sort_index(level='time')
)
df_grouped

value
model         split box         metric           root_ids     source            time             
Segformer_bce Test  230629PN008 Convex_Area_Hull (8, 1, 1)    Prediction        1      330.901647
                                                              before_expertized 1      328.799340
                                                              expertized        1      328.799340
                                                 (16, 11, 8)  Prediction        1      406.132539
                                                              before_expertized 1      406.477047
...                                                                                           ...
Unet_dice     Val   230629PN031 TotalRootLength  (68, 66, 63) before_expertized 29    2846.313710
                                                              expertized        29    2846.313710
                                                 (86, 84, 81) Prediction        29    3180.436231
                                                              before_expertized 29    3313.831728
                                                              expertized        29    3296.533489

[209496 rows x 1 columns]

In [4]:
models = df_grouped.index.get_level_values('model').unique()
splits = df_grouped.index.get_level_values('split').unique()
boxes = df_grouped.index.get_level_values('box').unique()
metrics = df_grouped.index.get_level_values('metric').unique()

print("Models disponibles :", models)
print("Splits disponibles :", splits)
print("Boxes disponibles  :", boxes)
print("Metrics disponibles:", metrics)

Models disponibles : Index(['Segformer_bce', 'Segformer_bce_dice', 'Segformer_dice', 'Unet_bce',
       'Unet_bce_dice', 'Unet_cldice_dice', 'Unet_dice'],
      dtype='object', name='model')
Splits disponibles : Index(['Test', 'Val'], dtype='object', name='split')
Boxes disponibles  : Index(['230629PN008', '230629PN010', '230629PN018', '230629PN024',
       '230629PN027', '230629PN006', '230629PN012', '230629PN014',
       '230629PN019', '230629PN031'],
      dtype='object', name='box')
Metrics disponibles: Index(['Convex_Area_Hull', 'Intercept_curve_Area', 'LateralRootLength',
       'NumberOfLateralRoots', 'NumberOfOrgans', 'PrimaryRootLength',
       'RootDensity', 'TotalRootLength'],
      dtype='object', name='metric')


In [5]:
def get_root_ids(model, split, box, metric):
    return (
        df_grouped
        .xs((model, split, box, metric), level=('model', 'split', 'box', 'metric'))
        .index.get_level_values('root_ids')
        .unique()
    )


def get_sources(model, split, box, metric, root):
    return (
        df_grouped
        .xs((model, split, box, metric, root), level=('model', 'split', 'box', 'metric', 'root_ids'))
        .index.get_level_values('source')
        .unique()
    )


def plot_root(model_idx, split_idx, box_idx, metric_idx, root_idx):
    model = models[model_idx]
    split = splits[split_idx]
    box = boxes[box_idx + split_idx * 5]
    metric = metrics[metric_idx]
    roots = get_root_ids(model, split, box, metric)
    root = roots[root_idx]

    df_tmp = (
        df_grouped
        .xs((model, split, box, metric, root),
            level=('model', 'split', 'box', 'metric', 'root_ids'))
        .reset_index()
    )
    #print(df_tmp.head())

    # Map styles + offset pour le jitter en abscisse
    style_map = {
        'before_expertized': dict(color='orange', linestyle='--', marker=None,
                                  linewidth=2.5, markersize=8, alpha=0.8, xoff=-0.1),
        'expertized': dict(color='green', linestyle='-', marker='o',
                           linewidth=2.5, markersize=8, alpha=0.8, xoff=0.0),
        'Prediction': dict(color='red', linestyle='-', marker='x',
                           linewidth=2.5, markersize=8, alpha=0.8, xoff=0.1),
    }

    plt.figure(figsize=(16, 8))
    for source, style in style_map.items():
        df_s = df_tmp[df_tmp['source'] == source]
        if df_s.empty:
            continue

        # convertir time en float si nécessaire
        x = pd.to_numeric(df_s['time'], errors='coerce').astype(float)
        # appliquer le jitter
        x = x + style['xoff']

        plt.plot(
            x, df_s['value'],
            label=source,
            color=style['color'],
            linestyle=style['linestyle'],
            marker=style['marker'],
            linewidth=style['linewidth'],
            markersize=style['markersize'],
            alpha=style['alpha']
        )

        # annoter le dernier point
        last_x = x.iloc[-1]
        last_y = df_s['value'].iloc[-1]
        plt.text(
            last_x, last_y,
            f" {source}",
            va='center',
            fontsize=9,
            color=style['color']
        )

    # Remettre les vraies valeurs de time en ticks
    uniq_times = sorted(df_tmp['time'].unique())
    plt.xticks(uniq_times, uniq_times)

    plt.title(f"{metric} – {model} / {split} / {box} / root {root}")
    plt.xlabel('Temps')
    plt.ylabel('Value')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(title='Source')
    plt.tight_layout()
    plt.show()

In [ ]:
model_slider = widgets.IntSlider(min=0, max=len(models) - 1, step=1, value=0, description='Model')
split_slider = widgets.IntSlider(min=0, max=len(splits) - 1, step=1, value=0, description='Split')
box_slider = widgets.IntSlider(min=0, max=4, step=1, value=0, description='Box')
metric_slider = widgets.IntSlider(min=0, max=len(metrics) - 1, step=1, value=0, description='Metric')

# Slider “root” : on fixe temporairement sa plage max à 0…9 – on mettra à jour dynamiquement
root_slider = widgets.IntSlider(min=0, max=0, step=1, value=0, description='Root')


# Callback pour mettre à jour le slider root_ids quand on change l’une des 4 premières valeurs
def update_root_slider(*args):
    roots = get_root_ids(
        models[model_slider.value],
        splits[split_slider.value],
        boxes[box_slider.value + split_slider.value * 5],
        metrics[metric_slider.value]
    )
    root_slider.max = len(roots) - 1
    if (root_slider.value >= root_slider.max):
        root_slider.value = root_slider.max


# On observe les changements
for w in (model_slider, split_slider, box_slider, metric_slider):
    w.observe(update_root_slider, 'value')

# Lancer la première initialisation
update_root_slider()

# 6) Affichage interactif
ui = widgets.HBox([model_slider, split_slider, box_slider, metric_slider, root_slider])
out = widgets.interactive_output(
    plot_root,
    {
        'model_idx': model_slider,
        'split_idx': split_slider,
        'box_idx': box_slider,
        'metric_idx': metric_slider,
        'root_idx': root_slider
    }
)

display(ui, out)

Output()

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

...

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

## Machin

In [7]:
df_plant_wid

,model,split,box,metric,time,root_ids,source,value
0,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",Prediction,1.000000
1,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",expertized,1.000000
2,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",before_expertized,1.000000
3,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(36, 47, 45)",Prediction,1.000000
4,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(36, 47, 45)",expertized,1.000000
...,...,...,...,...,...,...,...,...
209491,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(10, 14, 8)",expertized,125.584577
209492,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(10, 14, 8)",before_expertized,20.126268
209493,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(17, 23, 13)",Prediction,4.000000
209494,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(17, 23, 13)",expertized,119.834592


## normalized data frame

In [8]:
df_normalized = df_plant_wid.copy()
# Get the max value for each metric (regardless of time, model, or source)
max_value_for_each_metric = (
    df_normalized.groupby('metric')['value']
    .max()
    .reset_index()
    .rename(columns={'value': 'max_value'})
)

# Merge to assign max_value to each row
df_normalized = df_normalized.merge(max_value_for_each_metric, on='metric', how='left')
df_normalized['value'] = df_normalized['value'] / df_normalized['max_value']
df_normalized = df_normalized.drop(columns=['max_value'])
df_normalized

,model,split,box,metric,time,root_ids,source,value
0,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",Prediction,0.034483
1,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",expertized,0.034483
2,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(1, 1, 1)",before_expertized,0.034483
3,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(36, 47, 45)",Prediction,0.034483
4,Segformer_bce_dice,Val,230629PN031,NumberOfOrgans,1,"(36, 47, 45)",expertized,0.034483
...,...,...,...,...,...,...,...,...
209491,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(10, 14, 8)",expertized,0.037272
209492,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(10, 14, 8)",before_expertized,0.005973
209493,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(17, 23, 13)",Prediction,0.001187
209494,Unet_cldice_dice,Test,230629PN018,LateralRootLength,29,"(17, 23, 13)",expertized,0.035566


## Mean (not normalized) dataframe (mean for all plants)

In [9]:
# %% Agrégation par temps / model / metric
df_mean = (
    df_grouped
    .reset_index()  # remettre les colonnes
    .groupby(['model', 'metric', 'time', 'source'], as_index=False)
    .agg(mean_value=('value', 'mean'),
         std_value=('value', 'std')
         )
)

df_mean

,model,metric,time,source,mean_value,std_value
0,Segformer_bce,Convex_Area_Hull,1,Prediction,397.060906,67.128738
1,Segformer_bce,Convex_Area_Hull,1,before_expertized,390.886309,81.602448
2,Segformer_bce,Convex_Area_Hull,1,expertized,387.128963,39.024736
3,Segformer_bce,Convex_Area_Hull,2,Prediction,452.039324,72.314572
4,Segformer_bce,Convex_Area_Hull,2,before_expertized,443.585114,91.231965
...,...,...,...,...,...,...
4867,Unet_dice,TotalRootLength,28,before_expertized,1651.908930,916.730044
4868,Unet_dice,TotalRootLength,28,expertized,1817.790345,916.273673
4869,Unet_dice,TotalRootLength,29,Prediction,1768.862727,944.132276
4870,Unet_dice,TotalRootLength,29,before_expertized,1737.399024,960.907567


In [10]:
all_models = sorted(df_mean['model'].unique().tolist())

# styles fixes par source (identiques pour tous les modèles)
source_styles = {
    'Prediction': dict(linestyle='-', marker='o'),
}
source_order = ['Prediction']


def plot_mean_all_models(metric_idx):
    metric = metrics[metric_idx]  # slider sur la métrique uniquement
    df_plot = df_mean[df_mean['metric'] == metric].copy()
    if df_plot.empty:
        print(f"Aucune donnée pour la métrique {metric}.")
        return

    # s'assurer que time est numérique et trié
    df_plot['time'] = pd.to_numeric(df_plot['time'], errors='coerce')
    df_plot = df_plot.sort_values(['model', 'source', 'time'])

    plt.figure(figsize=(14, 7))

    # tracer chaque modèle avec une couleur différente,
    # et pour chaque source un style différent
    for m in all_models:
        d_m = df_plot[df_plot['model'] == m]
        if d_m.empty:
            continue
        for src in source_order:
            d = d_m[d_m['source'] == src]
            if d.empty:
                continue
            st = source_styles.get(src, dict(linestyle='-', marker=None))
            # on utilise le cycle de couleurs de Matplotlib: une couleur par modèle
            plt.plot(
                d['time'], d['mean_value'],
                label=f"{m} – {src}",
                linewidth=2.0, markersize=6, alpha=0.95,
                **st
            )

    plt.title(f"Évolution de la valeur moyenne – métrique: {metric}")
    plt.xlabel('Time')
    plt.ylabel('Mean value')
    plt.grid(True, linestyle=':', alpha=0.5)

    # Légende compacte
    plt.legend(title='Model – Source', fontsize=9, ncol=2)
    plt.tight_layout()
    plt.show()


# widget: slider uniquement sur la métrique
widgets.interactive(
    plot_mean_all_models,
    metric_idx=metric_slider
)


interactive(children=(IntSlider(value=0, description='Metric', max=7), Output()), _dom_classes=('widget-intera…

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

...

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

In [11]:
all_models = sorted(df_mean['model'].unique().tolist())

# styles fixes par source (identiques pour tous les modèles)
source_styles = {
    'Prediction': dict(linestyle='-', marker='o'),
}
source_order = ['Prediction']
colormap = plt.get_cmap("tab10")


def plot_mean_all_models(metric_idx):
    metric = metrics[metric_idx]  # slider sur la métrique uniquement
    df_plot = df_mean[df_mean['metric'] == metric].copy()
    if df_plot.empty:
        print(f"Aucune donnée pour la métrique {metric}.")
        return

    # s'assurer que time est numérique et trié
    df_plot['time'] = pd.to_numeric(df_plot['time'], errors='coerce')
    df_plot = df_plot.sort_values(['model', 'source', 'time'])

    plt.figure(figsize=(14, 7))

    # tracer chaque modèle avec une couleur différente,
    # et pour chaque source un style différent
    for m, style in zip(all_models, colormap.colors):
        d_m = df_plot[df_plot['model'] == m]
        if d_m.empty:
            continue
        for src in source_order:
            d = d_m[d_m['source'] == src]
            if d.empty:
                continue
            st = source_styles.get(src, dict(linestyle='-', marker=None))
            # on utilise le cycle de couleurs de Matplotlib: une couleur par modèle
            plt.plot(
                d['time'], d['mean_value'],
                label=f"{m} – {src}",
                linestyle=st['linestyle'], marker=st['marker'],
                color=style
            )
            # std
            plt.fill_between(
                d['time'], d['mean_value'] - d['std_value'],
                           d['mean_value'] + d['std_value'],
                alpha=0.2, color=style
            )

    plt.title(f"Évolution de la valeur moyenne – métrique: {metric}")
    plt.xlabel('Time')
    plt.ylabel('Mean value')
    plt.grid(True, linestyle=':', alpha=0.5)

    # Légende compacte
    plt.legend(title='Model – Source', fontsize=9, ncol=2)
    plt.tight_layout()
    plt.show()


# widget: slider uniquement sur la métrique
widgets.interactive(
    plot_mean_all_models,
    metric_idx=metric_slider
)

interactive(children=(IntSlider(value=0, description='Metric', max=7), Output()), _dom_classes=('widget-intera…

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

...

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

In [ ]:
# for each model and metric, plot the evolution over time of mean value

def plot_mean_evolution(model_idx, metric_idx):
    model = models[model_idx]
    metric = metrics[metric_idx]
    df_plot = df_mean[(df_mean['model'] == model) & (df_mean['metric'] == metric)]
    plt.figure(figsize=(16, 10))
    for source, style in zip(['Prediction', 'before_expertized', 'expertized'],
                             [{'color': 'red', 'linestyle': '-', 'marker': 'x'},
                              {'color': 'orange', 'linestyle': '--', 'marker': None},
                              {'color': 'green', 'linestyle': '-', 'marker': 'o'}]):
        df_s = df_plot[df_plot['source'] == source]
        if df_s.empty:
            continue
        plt.errorbar(
            df_s['time'], df_s['mean_value'],
            label=source,
            color=style['color'],
            linestyle=style['linestyle'],
            marker=style['marker'],
            linewidth=2.5,
            markersize=8,
            alpha=0.8
        )
        plt.fill_between(
            df_s['time'], df_s['mean_value'] - df_s['std_value'],
                          df_s['mean_value'] + df_s['std_value'],
            alpha=0.2, color=style['color']
        )
    plt.title(f"Mean evolution over time – {model} / {metric}")
    plt.xlabel('Time')
    plt.ylabel('Mean Value')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(title='Source')
    plt.tight_layout()
    plt.show()


widgets.interactive(
    plot_mean_evolution,
    model_idx=model_slider,
    metric_idx=metric_slider
)



interactive(children=(IntSlider(value=0, description='Model', max=6), IntSlider(value=0, description='Metric',…

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

...

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

## Error

In [13]:
# Pivot the dataframe to get 'Prediction', 'expertized', and 'real' as columns
df_pivot = (
    df_plant_wid
    .pivot_table(
        index=['model', 'metric', 'time', 'split', 'box', 'root_ids'],
        columns='source',
        values='value'
    )
    .reset_index()
)

# Compute errors only if both columns exist
df_pivot['abs_error'] = (df_pivot['expertized'] - df_pivot['Prediction']).abs()
df_pivot['real_error'] = df_pivot['expertized'] - df_pivot['Prediction']

df_error = df_pivot[
    ['model', 'metric', 'time', 'split', 'box', 'root_ids', 'abs_error', 'real_error']
]
df_error

source,model,metric,time,split,box,root_ids,abs_error,real_error
0,Segformer_bce,Convex_Area_Hull,1,Test,230629PN008,"(8, 1, 1)",2.102307,-2.102307
1,Segformer_bce,Convex_Area_Hull,1,Test,230629PN008,"(16, 11, 8)",0.344508,0.344508
2,Segformer_bce,Convex_Area_Hull,1,Test,230629PN010,"(15, 1, 3)",0.066454,-0.066454
3,Segformer_bce,Convex_Area_Hull,1,Test,230629PN010,"(26, 16, 14)",0.693757,0.693757
4,Segformer_bce,Convex_Area_Hull,1,Test,230629PN010,"(44, 35, 32)",0.016828,-0.016828
...,...,...,...,...,...,...,...,...
69827,Unet_dice,TotalRootLength,29,Val,230629PN031,"(1, 1, 1)",129.536772,129.536772
69828,Unet_dice,TotalRootLength,29,Val,230629PN031,"(20, 19, 18)",497.480324,497.480324
69829,Unet_dice,TotalRootLength,29,Val,230629PN031,"(48, 47, 45)",134.087492,134.087492
69830,Unet_dice,TotalRootLength,29,Val,230629PN031,"(68, 66, 63)",3.381529,-3.381529


In [14]:
df_mean_error = (
    df_error
    .groupby(['model', 'metric', 'time'], as_index=False)
    .agg(
        abs_error_mean=('abs_error', 'mean'),
        real_error_mean=('real_error', 'mean'),
        std_abs_error=('abs_error', 'std'),
    )
)
df_mean_error

,model,metric,time,abs_error_mean,real_error_mean,std_abs_error
0,Segformer_bce,Convex_Area_Hull,1,12.555731,-9.931943,49.186977
1,Segformer_bce,Convex_Area_Hull,2,12.400503,-10.519181,48.641174
2,Segformer_bce,Convex_Area_Hull,3,18.788374,-4.594231,61.484383
3,Segformer_bce,Convex_Area_Hull,4,18.234050,-4.403006,61.262677
4,Segformer_bce,Convex_Area_Hull,5,18.436631,-4.040144,61.956750
...,...,...,...,...,...,...
1619,Unet_dice,TotalRootLength,25,93.149819,77.794890,100.333960
1620,Unet_dice,TotalRootLength,26,107.247941,93.108750,105.980646
1621,Unet_dice,TotalRootLength,27,123.805594,109.110001,116.809601
1622,Unet_dice,TotalRootLength,28,142.863203,127.876818,124.589044


In [15]:
all_models = sorted(df_mean_error['model'].unique().tolist())


def plot_distance_all_models(metric_idx):
    metric = metrics[metric_idx]
    df_plot = df_mean_error[df_mean_error['metric'] == metric].copy()
    if df_plot.empty:
        print(f"Aucune donnée pour la métrique {metric}.")
        return

    df_plot['time'] = pd.to_numeric(df_plot['time'], errors='coerce')
    df_plot = df_plot.sort_values(['model', 'time'])

    plt.figure(figsize=(14, 7))
    for m in all_models:
        d = df_plot[df_plot['model'] == m]
        if d.empty:
            continue
        plt.plot(
            d['time'], d['abs_error_mean'],
            marker='o', linewidth=2.0, markersize=6, alpha=0.95, label=m
        )

    plt.title(f"Évolution de l'erreur moyenne – métrique: {metric}")
    plt.xlabel('Time')
    plt.ylabel('Mean relative error')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(title='Model', ncol=2, fontsize=9)
    plt.tight_layout()
    plt.show()


# on garde le slider métrique existant
widgets.interactive(
    plot_distance_all_models,
    metric_idx=metric_slider
)


interactive(children=(IntSlider(value=0, description='Metric', max=7), Output()), _dom_classes=('widget-intera…

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

...

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

## Each model mean error normalization

In [16]:
df_max_value_for_each_metric = (
    df_mean_error
    .groupby(['metric'])['abs_error_mean']
    .max()
)

df_max_value_for_each_metric

metric
Convex_Area_Hull         941.830607
Intercept_curve_Area       2.709337
LateralRootLength        515.469565
NumberOfLateralRoots       7.581395
NumberOfOrgans             7.581395
PrimaryRootLength        499.424256
RootDensity                0.232979
TotalRootLength         1014.893821
Name: abs_error_mean, dtype: float64

In [22]:
df_normalized = df_mean_error.copy()
# Create a MultiIndex for mapping
df_normalized['max_error'] = df_normalized.set_index(['metric']).index.map(df_max_value_for_each_metric)
df_normalized['abs_error_mean'] = 1 - df_normalized['abs_error_mean'] / df_normalized['max_error']
df_normalized['real_error_mean'] = df_normalized['real_error_mean'] / df_normalized['max_error']
df_normalized['std_abs_error'] = df_normalized['std_abs_error'] / df_normalized['max_error']
df_normalized = df_normalized.drop(columns=['max_error'])

# rename abs_error_mean to normalized_abs_error_mean
df_normalized = df_normalized.rename(columns={'abs_error_mean': 'norm_abs_approx'})
df_normalized

,model,metric,time,norm_abs_approx,real_error_mean,std_abs_error
0,Segformer_bce,Convex_Area_Hull,1,0.986669,-0.010545,0.052225
1,Segformer_bce,Convex_Area_Hull,2,0.986834,-0.011169,0.051645
2,Segformer_bce,Convex_Area_Hull,3,0.980051,-0.004878,0.065282
3,Segformer_bce,Convex_Area_Hull,4,0.980640,-0.004675,0.065046
4,Segformer_bce,Convex_Area_Hull,5,0.980425,-0.004290,0.065783
...,...,...,...,...,...,...
1619,Unet_dice,TotalRootLength,25,0.908217,0.076653,0.098862
1620,Unet_dice,TotalRootLength,26,0.894326,0.091742,0.104425
1621,Unet_dice,TotalRootLength,27,0.878011,0.107509,0.115095
1622,Unet_dice,TotalRootLength,28,0.859233,0.126000,0.122761


In [27]:
# %% Optionnel : visualisation de l'évolution de la distance moyenne au cours du temps

def plot_distance_evolution(model_idx, metric_idx):
    model = models[model_idx]
    metric = metrics[metric_idx]
    df_plot = df_normalized[(df_normalized['model'] == model) & (df_normalized['metric'] == metric)]
    if df_plot.empty:
        print("Aucune paire Prediction/expertized trouvée pour cette combinaison.")
        return
    plt.figure(figsize=(14, 7))
    plt.errorbar(
        df_plot['time'], df_plot['norm_abs_approx'],
        label='|Prediction - expertized| / max_{metric, time} (|Prediction - expertized|)',
        linestyle='-', marker='o', linewidth=2.5, markersize=7, alpha=0.9
    )
    plt.fill_between(
        df_plot['time'],
        df_plot['norm_abs_approx'] - df_plot['std_abs_error'],
        df_plot['norm_abs_approx'] + df_plot['std_abs_error'],
        alpha=0.2, color='blue'
    )
    plt.title(f"Évolution de l'erreur absolue normalisée – {model} / {metric}")
    plt.xlabel('Time')
    plt.ylabel('Norm. mean absolute error')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend()
    plt.ylim(0, 1.2)
    plt.tight_layout()
    plt.show()


widgets.interactive(
    plot_distance_evolution,
    model_idx=model_slider,
    metric_idx=metric_slider
)


interactive(children=(IntSlider(value=3, description='Model', max=6), IntSlider(value=4, description='Metric',…

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

...

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

In [28]:
def plot_distance_all_models(metric_idx):
    metric = metrics[metric_idx]
    df_plot = df_normalized[df_normalized['metric'] == metric].copy()
    if df_plot.empty:
        print(f"Aucune donnée pour la métrique '{metric}'.")
        return

    # on s'assure que time est numérique et trié par modèle
    df_plot['time'] = pd.to_numeric(df_plot['time'], errors='coerce')
    df_plot = df_plot.sort_values(['model', 'time'])

    # couleurs auto (une par modèle)
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    models_in_plot = df_plot['model'].unique().tolist()

    plt.figure(figsize=(14, 7))
    for i, m in enumerate(models_in_plot):
        d = df_plot[df_plot['model'] == m]
        if d.empty:
            continue

        # courbe + barres d'erreur (plus lisible que des aplats multiples)
        plt.errorbar(
            d['time'], d['norm_abs_approx'],
            yerr=d.get('std_abs_error', None),
            label=m,
            linestyle='-',
            marker='o',
            linewidth=2.0,
            markersize=6,
            alpha=0.9,
            color=colors[i % len(colors)],
            capsize=3,
        )

    plt.title(f"Évolution de l'erreur absolue normalisée — métrique: {metric}")
    plt.xlabel('Time')
    plt.ylabel('Norm. mean absolute error')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(title='Model', ncol=2, fontsize=9)
    plt.ylim(0, 1)  # fixe l'ordonnée entre 0 et 1
    plt.tight_layout()
    plt.show()


# Slider uniquement pour la métrique
widgets.interactive(
    plot_distance_all_models,
    metric_idx=metric_slider
)

interactive(children=(IntSlider(value=4, description='Metric', max=7), Output()), _dom_classes=('widget-intera…

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

<Figure size 1400x700 with 0 Axes>

## Error at all time

In [30]:
df_norm_mean_at_all_time = df_normalized.groupby(['metric', 'model'])['norm_abs_approx'].mean()
df_norm_mean_at_all_time

metric                model             
Convex_Area_Hull      Segformer_bce         0.978751
                      Segformer_bce_dice    0.977429
                      Segformer_dice        0.978883
                      Unet_bce              0.977702
                      Unet_bce_dice         0.641758
                      Unet_cldice_dice      0.978623
                      Unet_dice             0.979299
Intercept_curve_Area  Segformer_bce         0.934294
                      Segformer_bce_dice    0.927168
                      Segformer_dice        0.934339
                      Unet_bce              0.940036
                      Unet_bce_dice         0.713188
                      Unet_cldice_dice      0.933357
                      Unet_dice             0.949071
LateralRootLength     Segformer_bce         0.892679
                      Segformer_bce_dice    0.877193
                      Segformer_dice        0.890859
                      Unet_bce              0.901661
     

In [16]:
# %% Distance moyenne Prediction vs expertized (par model, metric, time)

# On repart du long format
df_long = df_grouped.reset_index()  # colonnes: model, split, box, metric, root_ids, source, time, value

# Séparer les deux sources que l'on veut comparer
df_pred = df_long[df_long['source'] == 'Prediction'].copy()
df_exp = df_long[df_long['source'] == 'before_expertized'].copy()

# Clés d'appariement: même instance (même model/metric/temps/plant)
keys = ['model', 'metric', 'time', 'split', 'box', 'root_ids']

# Fusion 1–1 entre Prediction et expertized sur les mêmes observations
paired = df_pred.merge(
    df_exp,
    on=keys,
    how='inner',
    suffixes=('_pred', '_exp')
)

# Si time est du texte numérique, convertir en float (facultatif)
paired['time'] = pd.to_numeric(paired['time'], errors='ignore')

# Distance absolue
paired['diff'] = (paired['value_pred'] - paired['value_exp']).abs()
# for each metric, get max exp value 
paired['max_exp_value'] = paired.groupby(keys)['value_exp'].transform('max')
paired['ratio'] = paired['diff'] / paired['max_exp_value']

# Agrégation demandée : moyenne par model/metric/time
df_dist = (
    paired
    .groupby(['model', 'metric', 'time'], as_index=False)
    .agg(
        mean_distance=('diff', 'mean'),
        max_expertized_value=('max_exp_value', 'mean'),
        mean_ratio=('ratio', 'mean')
    )
    .sort_values(['model', 'metric', 'time'])
)

df_dist


/tmp/ipykernel_153262/1630556097.py:24: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  paired['time'] = pd.to_numeric(paired['time'], errors='ignore')


,model,metric,time,mean_distance,max_expertized_value,mean_ratio
0,Segformer_bce,Convex_Area_Hull,1,11.282043,390.886309,0.105540
1,Segformer_bce,Convex_Area_Hull,2,11.995991,443.585114,0.117114
2,Segformer_bce,Convex_Area_Hull,3,12.628666,476.877477,0.123430
3,Segformer_bce,Convex_Area_Hull,4,12.943969,510.282559,0.129107
4,Segformer_bce,Convex_Area_Hull,5,13.709903,539.369409,0.136640
...,...,...,...,...,...,...
1619,Unet_dice,TotalRootLength,25,104.230646,1310.447026,0.133551
1620,Unet_dice,TotalRootLength,26,114.642658,1417.396285,0.132979
1621,Unet_dice,TotalRootLength,27,129.897614,1533.743351,0.135342
1622,Unet_dice,TotalRootLength,28,141.528506,1651.908930,0.136049


In [17]:
# %% Optionnel : visualisation de l'évolution de la distance moyenne au cours du temps

def plot_distance_evolution(model_idx, metric_idx):
    model = models[model_idx]
    metric = metrics[metric_idx]
    df_plot = df_dist[(df_dist['model'] == model) & (df_dist['metric'] == metric)]
    if df_plot.empty:
        print("Aucune paire Prediction/expertized trouvée pour cette combinaison.")
        return
    plt.figure(figsize=(14, 7))
    plt.errorbar(
        df_plot['time'], df_plot['mean_ratio'],
        label='|Prediction - expertized|',
        linestyle='-', marker='o', linewidth=2.5, markersize=7, alpha=0.9
    )
    plt.title(f"Erreur absolue Prediction vs expertized – {model} / {metric}")
    plt.xlabel('Time')
    plt.ylabel('Mean absolute error')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()


widgets.interactive(
    plot_distance_evolution,
    model_idx=model_slider,
    metric_idx=metric_slider
)


interactive(children=(IntSlider(value=0, description='Model', max=6), IntSlider(value=0, description='Metric',…

In [18]:
# On part de df_mean: colonnes ['model', 'metric', 'time', 'source', 'mean_value', 'std_value']

# 1) On ne garde que les deux sources utiles
df_pe = df_mean[df_mean['source'].isin(['Prediction', 'expertized'])].copy()

# 2) Pivot pour aligner Prediction et expertized sur chaque (model, metric, time)
pe_pivot = (
    df_pe
    .pivot_table(index=['model', 'metric', 'time'],
                 columns='source',
                 values='mean_value',  # on compare les moyennes agrégées
                 aggfunc='mean')  # au cas où il y aurait des doublons
    .reset_index()
)

# 3) Calcul de la différence par instant (time)
pe_pivot['diff_signed'] = pe_pivot['Prediction'] - pe_pivot['expertized']
pe_pivot['diff_abs'] = pe_pivot['diff_signed'].abs()

pe_pivot


source,model,metric,time,Prediction,expertized,diff_signed,diff_abs
0,Segformer_bce,Convex_Area_Hull,1,397.060906,387.128963,9.931943,9.931943
1,Segformer_bce,Convex_Area_Hull,2,452.039324,441.520143,10.519181,10.519181
2,Segformer_bce,Convex_Area_Hull,3,486.219655,481.625424,4.594231,4.594231
3,Segformer_bce,Convex_Area_Hull,4,520.280929,515.877923,4.403006,4.403006
4,Segformer_bce,Convex_Area_Hull,5,550.085070,546.044927,4.040144,4.040144
...,...,...,...,...,...,...,...
1619,Unet_dice,TotalRootLength,25,1348.283773,1426.078663,-77.794890,77.794890
1620,Unet_dice,TotalRootLength,26,1456.002025,1549.110774,-93.108750,93.108750
1621,Unet_dice,TotalRootLength,27,1574.769257,1683.879259,-109.110001,109.110001
1622,Unet_dice,TotalRootLength,28,1689.913526,1817.790345,-127.876818,127.876818


In [19]:
# 4) Moyenne (dans le temps) par (model, metric)
df_diff_by_model_metric = (
    pe_pivot
    .groupby(['model', 'metric'], as_index=False)
    .agg(
        mean_abs_diff=('diff_abs', 'mean'),
        mean_signed_diff=('diff_signed', 'mean')
    )
    .sort_values(['metric', 'model'])
)

df_diff_by_model_metric

,model,metric,mean_abs_diff,mean_signed_diff
0,Segformer_bce,Convex_Area_Hull,3.331662,1.687884
8,Segformer_bce_dice,Convex_Area_Hull,4.806729,4.157522
16,Segformer_dice,Convex_Area_Hull,3.166904,2.626064
24,Unet_bce,Convex_Area_Hull,5.292291,0.927214
32,Unet_bce_dice,Convex_Area_Hull,328.875616,-328.137397
40,Unet_cldice_dice,Convex_Area_Hull,4.742116,3.083674
48,Unet_dice,Convex_Area_Hull,4.198218,2.705298
1,Segformer_bce,Intercept_curve_Area,0.139197,-0.130119
9,Segformer_bce_dice,Intercept_curve_Area,0.164315,-0.155038
17,Segformer_dice,Intercept_curve_Area,0.138929,-0.129913


In [20]:
all_models = sorted(df_mean['model'].unique().tolist())
all_models = ['Segformer_bce', 'Segformer_bce_dice', 'Segformer_dice', 'Unet_bce', 'Unet_cldice_dice',
              'Unet_dice']  # , 'Unet_bce_dice']
print(all_models)

['Segformer_bce', 'Segformer_bce_dice', 'Segformer_dice', 'Unet_bce', 'Unet_cldice_dice', 'Unet_dice']


In [21]:
all_models_to_f1 = {m: 0 for m in all_models}
all_models_to_precision = {m: 0 for m in all_models}
all_models_to_recall = {m: 0 for m in all_models}
all_models_to_iou = {m: 0 for m in all_models}
all_models_to_acd = {m: 0 for m in all_models}
all_models_to_dice = {m: 0 for m in all_models}
all_models_to_alps = {m: 0 for m in all_models}
all_models_to_betti0 = {m: 0 for m in all_models}
all_models_to_betti1 = {m: 0 for m in all_models}
all_models_to_haussdorf = {m: 0 for m in all_models}
all_models_to_persistence0 = {m: 0 for m in all_models}
all_models_to_persistence1 = {m: 0 for m in all_models}

all_models_to_f1['Segformer_bce'] = 0.91804111
all_models_to_alps['Segformer_bce'] = 0.8555116653
all_models_to_betti0['Segformer_bce'] = 0.1803682894
all_models_to_betti1['Segformer_bce'] = 0.1659309715
all_models_to_haussdorf['Segformer_bce'] = 32.4457283
all_models_to_persistence0['Segformer_bce'] = 3.200774431
all_models_to_persistence1['Segformer_bce'] = 73.74468231

all_models_to_f1['Segformer_bce_dice'] = 0.9048361778
all_models_to_alps['Segformer_bce_dice'] = 0.8482627273
all_models_to_betti0['Segformer_bce_dice'] = 0.1853652745
all_models_to_betti1['Segformer_bce_dice'] = 0.154604733
all_models_to_haussdorf['Segformer_bce_dice'] = 32.02287292
all_models_to_persistence0['Segformer_bce_dice'] = 3.07885313
all_models_to_persistence1['Segformer_bce_dice'] = 130.1295776

all_models_to_f1['Segformer_dice'] = 0.9168285131
all_models_to_alps['Segformer_dice'] = 0.8827118874
all_models_to_betti0['Segformer_dice'] = 0.1841645688
all_models_to_betti1['Segformer_dice'] = 0.1399480104
all_models_to_haussdorf['Segformer_dice'] = 31.13061523
all_models_to_persistence0['Segformer_dice'] = 3.512531519
all_models_to_persistence1['Segformer_dice'] = 133.3082581

all_models_to_f1['Unet_bce'] = 0.9468662739
all_models_to_alps['Unet_bce'] = 0.9345962405
all_models_to_betti0['Unet_bce'] = 0.1148660257
all_models_to_betti1['Unet_bce'] = 0.1221347302
all_models_to_haussdorf['Unet_bce'] = 29.71125793
all_models_to_persistence0['Unet_bce'] = 1.714779496
all_models_to_persistence1['Unet_bce'] = 53.67120743

all_models_to_f1['Unet_bce_dice'] = 0.9287698269
all_models_to_alps['Unet_bce_dice'] = 0.4729110897
all_models_to_betti0['Unet_bce_dice'] = 0.3485465646
all_models_to_betti1['Unet_bce_dice'] = 0.1686320454
all_models_to_haussdorf['Unet_bce_dice'] = 31.91540146
all_models_to_persistence0['Unet_bce_dice'] = 2.485318661
all_models_to_persistence1['Unet_bce_dice'] = 65.56051636

all_models_to_f1['Unet_cldice_dice'] = 0.9485918283
all_models_to_alps['Unet_cldice_dice'] = 0.938965559
all_models_to_betti0['Unet_cldice_dice'] = 0.118898496
all_models_to_betti1['Unet_cldice_dice'] = 0.1722722054
all_models_to_haussdorf['Unet_cldice_dice'] = 29.60779953
all_models_to_persistence0['Unet_cldice_dice'] = 1.79317379
all_models_to_persistence1['Unet_cldice_dice'] = 56.66341019

all_models_to_f1['Unet_dice'] = 0.9471697807
all_models_to_alps['Unet_dice'] = 0.9078174233
all_models_to_betti0['Unet_dice'] = 0.1029598936
all_models_to_betti1['Unet_dice'] = 0.1002777293
all_models_to_haussdorf['Unet_dice'] = 30.6745224
all_models_to_persistence0['Unet_dice'] = 1.776620388
all_models_to_persistence1['Unet_dice'] = 57.75206757


In [22]:
import numpy as np
import pandas as pd
import ipywidgets as widgets

# ---------- 1) Table des métriques externes par modèle ----------
external_metrics = pd.DataFrame({
    'model': all_models,
    'f1': [all_models_to_f1[m] for m in all_models],  # score
    'alps': [all_models_to_alps[m] for m in all_models],  # score
    'betti0': [all_models_to_betti0[m] for m in all_models],  # distance
    'betti1': [all_models_to_betti1[m] for m in all_models],  # distance
    'hausdorff': [all_models_to_haussdorf[m] for m in all_models],  # distance
    'persistence0': [all_models_to_persistence0[m] for m in all_models],  # distance
    'persistence1': [all_models_to_persistence1[m] for m in all_models],  # distance
})

# Familles / catégories
categories = {
    "graph extracted": ["f1", 'hausdorff', "alps", "betti0", "betti1", "persistence0", "persistence1"],
    "computer vision": ["f1", "hausdorff"],
}

category_type = {
    "f1": "score",
    "hausdorff": "distance",
    "alps": "score",
    "betti0": "distance",
    "betti1": "distance",
    "persistence0": "distance",
    "persistence1": "distance",
}


# ---------- 2) Fonction de tracé ----------
def plot_small_multiples(category, x_metric_key):
    # sanity: s'assurer que le x_metric_key vient bien de la catégorie choisie
    if x_metric_key not in categories[category]:
        print(f"La métrique '{x_metric_key}' n'appartient pas à la catégorie '{category}'.")
        return

    # jeux de couleurs: un par modèle
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    color_map = {m: colors[i % len(colors)] for i, m in enumerate(all_models)}

    # Liste des mesures internes (ordonnées) disponibles
    inner_measures = sorted(df_diff_by_model_metric['metric'].unique().tolist())

    # Layout des subplots
    n = len(inner_measures)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.8 * nrows), squeeze=False)
    fig.suptitle(f"Y: mean_abs_diff (par modèle & mesure)  |  X: {x_metric_key}  [{category}]", fontsize=14)

    # Data externe (X) par modèle
    x_by_model = dict(zip(external_metrics['model'], external_metrics[x_metric_key]))

    # Pour une échelle couleur cohérente en legende, on garde un handle par modèle
    legend_handles = []
    handled_models = set()

    # Tracé par subplot (une mesure interne = un panel)
    for idx, meas in enumerate(inner_measures):
        ax = axes[idx // ncols, idx % ncols]

        # Sous-DF (une mesure interne donnée)
        d_meas = df_diff_by_model_metric[df_diff_by_model_metric['metric'] == meas]

        # Scatter par modèle
        for m in all_models:
            d_m = d_meas[d_meas['model'] == m]
            if d_m.empty or m not in x_by_model:
                continue

            # y = mean_abs_diff (un point par modèle pour cette mesure)
            y = d_m['mean_abs_diff'].values
            # x = métrique externe du modèle (scalaire), répété au besoin
            x = np.repeat(x_by_model[m], len(y))

            c = color_map[m]
            sc = ax.scatter(x, y, label=m if m not in handled_models else None, s=30, alpha=0.9, color=c)
            # invert y axis
            # if metric as f1 is a score, do not invert x axis. otherwise do it
            if category_type[x_metric_key] == "distance":
                ax.invert_xaxis()
            ax.invert_yaxis()

            if m not in handled_models:
                legend_handles.append(sc)
                handled_models.add(m)

        ax.set_title(meas)
        ax.set_xlabel(x_metric_key)
        ax.set_ylabel("mean_abs_diff")
        ax.grid(True, linestyle=":", alpha=0.5)

    # Nettoyage des axes vides (si pas multiple de ncols)
    for k in range(n, nrows * ncols):
        fig.delaxes(axes[k // ncols, k % ncols])

    # Légende globale des modèles
    if legend_handles:
        fig.legend(handles=legend_handles, labels=[h.get_label() for h in legend_handles],
                   title="Model", ncol=min(len(all_models), 4),
                   loc="upper right", bbox_to_anchor=(0.98, 0.98))

    plt.tight_layout(rect=[0, 0, 0.92, 0.96])
    plt.show()


# ---------- 3) Widgets ----------
category_slider = widgets.SelectionSlider(
    options=list(categories.keys()),
    value="graph extracted",
    description='Famille:',
    continuous_update=False
)


def x_metric_options(category):
    return categories[category]


x_metric_slider = widgets.SelectionSlider(
    options=x_metric_options(category_slider.value),
    value=x_metric_options(category_slider.value)[0],
    description='X metric:',
    continuous_update=False
)


def update_x_options(change):
    x_metric_slider.options = x_metric_options(change['new'])
    x_metric_slider.value = x_metric_options(change['new'])[0]


category_slider.observe(update_x_options, names='value')

widgets.interactive(
    plot_small_multiples,
    category=category_slider,
    x_metric_key=x_metric_slider
)


interactive(children=(SelectionSlider(continuous_update=False, description='Famille:', options=('graph extract…

In [23]:
df_diff_by_model_metric


,model,metric,mean_abs_diff,mean_signed_diff
0,Segformer_bce,Convex_Area_Hull,3.331662,1.687884
8,Segformer_bce_dice,Convex_Area_Hull,4.806729,4.157522
16,Segformer_dice,Convex_Area_Hull,3.166904,2.626064
24,Unet_bce,Convex_Area_Hull,5.292291,0.927214
32,Unet_bce_dice,Convex_Area_Hull,328.875616,-328.137397
40,Unet_cldice_dice,Convex_Area_Hull,4.742116,3.083674
48,Unet_dice,Convex_Area_Hull,4.198218,2.705298
1,Segformer_bce,Intercept_curve_Area,0.139197,-0.130119
9,Segformer_bce_dice,Intercept_curve_Area,0.164315,-0.155038
17,Segformer_dice,Intercept_curve_Area,0.138929,-0.129913


In [24]:
df_scores = pd.DataFrame({
    "model": sorted(df_diff_by_model_metric['model'].unique().tolist()),
    "f1": [all_models_to_f1[model] for model in all_models_to_f1.keys()],
    "hausdorff": [all_models_to_haussdorf[model] for model in all_models_to_haussdorf.keys()],
    "alps": [all_models_to_alps[model] for model in all_models_to_alps.keys()],
    "betti0": [all_models_to_betti0[model] for model in all_models_to_betti0.keys()],
    "betti1": [all_models_to_betti1[model] for model in all_models_to_betti1.keys()],
    "persistence0": [all_models_to_persistence0[model] for model in all_models_to_persistence0.keys()],
    "persistence1": [all_models_to_persistence1[model] for model in all_models_to_persistence1.keys()],
})

df_scores

,model,f1,hausdorff,alps,betti0,betti1,persistence0,persistence1
0,Segformer_bce,0.918041,32.445728,0.855512,0.180368,0.165931,3.200774,73.744682
1,Segformer_bce_dice,0.904836,32.022873,0.848263,0.185365,0.154605,3.078853,130.129578
2,Segformer_dice,0.916829,31.130615,0.882712,0.184165,0.139948,3.512532,133.308258
3,Unet_bce,0.946866,29.711258,0.934596,0.114866,0.122135,1.714779,53.671207
4,Unet_bce_dice,0.948592,29.607800,0.938966,0.118898,0.172272,1.793174,56.663410
5,Unet_cldice_dice,0.947170,30.674522,0.907817,0.102960,0.100278,1.776620,57.752068
6,Unet_dice,0.928770,31.915401,0.472911,0.348547,0.168632,2.485319,65.560516


In [25]:
# add columns "f1", "hausdorff", "alps", "betti0", "betti1", "persistence0", "persistence1" to df_diff_by_model_metric
# using all_models_to_X, for each model, give its score as a column value
df_diff_by_model_metric["f1"] = df_diff_by_model_metric["model"].map(all_models_to_f1)
df_diff_by_model_metric["hausdorff"] = df_diff_by_model_metric["model"].map(all_models_to_haussdorf)
df_diff_by_model_metric["alps"] = df_diff_by_model_metric["model"].map(all_models_to_alps)
df_diff_by_model_metric["betti0"] = df_diff_by_model_metric["model"].map(all_models_to_betti0)
df_diff_by_model_metric["betti1"] = df_diff_by_model_metric["model"].map(all_models_to_betti1)
df_diff_by_model_metric["persistence0"] = df_diff_by_model_metric["model"].map(all_models_to_persistence0)
df_diff_by_model_metric["persistence1"] = df_diff_by_model_metric["model"].map(all_models_to_persistence1)
df_diff_by_model_metric

,model,metric,mean_abs_diff,mean_signed_diff,f1,hausdorff,alps,betti0,betti1,persistence0,persistence1
0,Segformer_bce,Convex_Area_Hull,3.331662,1.687884,0.918041,32.445728,0.855512,0.180368,0.165931,3.200774,73.744682
8,Segformer_bce_dice,Convex_Area_Hull,4.806729,4.157522,0.904836,32.022873,0.848263,0.185365,0.154605,3.078853,130.129578
16,Segformer_dice,Convex_Area_Hull,3.166904,2.626064,0.916829,31.130615,0.882712,0.184165,0.139948,3.512532,133.308258
24,Unet_bce,Convex_Area_Hull,5.292291,0.927214,0.946866,29.711258,0.934596,0.114866,0.122135,1.714779,53.671207
32,Unet_bce_dice,Convex_Area_Hull,328.875616,-328.137397,0.928770,31.915401,0.472911,0.348547,0.168632,2.485319,65.560516
40,Unet_cldice_dice,Convex_Area_Hull,4.742116,3.083674,0.948592,29.607800,0.938966,0.118898,0.172272,1.793174,56.663410
48,Unet_dice,Convex_Area_Hull,4.198218,2.705298,0.947170,30.674522,0.907817,0.102960,0.100278,1.776620,57.752068
1,Segformer_bce,Intercept_curve_Area,0.139197,-0.130119,0.918041,32.445728,0.855512,0.180368,0.165931,3.200774,73.744682
9,Segformer_bce_dice,Intercept_curve_Area,0.164315,-0.155038,0.904836,32.022873,0.848263,0.185365,0.154605,3.078853,130.129578
17,Segformer_dice,Intercept_curve_Area,0.138929,-0.129913,0.916829,31.130615,0.882712,0.184165,0.139948,3.512532,133.308258


In [ ]:
# %% [markdown]
# ## Radar des mean_abs_diff par modèle (normalisé par métrique)

import numpy as np
import ipywidgets as widgets


# On repart de df_diff_by_model_metric : [model, metric, mean_abs_diff, mean_signed_diff, ...]
# -> Normalisation min-max PAR métrique sur l'ensemble des modèles
def _minmax_inv(x_min, x_max, x):
    # retourne 1 - (x - min) / (max - min) ; si max==min -> 1
    if not np.isfinite(x_min) or not np.isfinite(x_max) or np.isclose(x_min, x_max):
        return 1.0
    return float(1.0 - (x - x_min) / (x_max - x_min))


# Stats par métrique
stats = (
    df_diff_by_model_metric
    .groupby('metric', as_index=False)['mean_abs_diff']
    .agg(metric_min='min', metric_max='max')
)

# Merge pour avoir min/max sur chaque ligne
df_norm = df_diff_by_model_metric.merge(stats, on='metric', how='left').copy()
df_norm['score_norm'] = [
    _minmax_inv(r.metric_min, r.metric_max, r.mean_abs_diff) for r in df_norm.itertuples()
]
# score_norm ∈ [0,1] où 1 = meilleur (erreur faible), 0 = pire (erreur forte)

# Liste ordonnée des métriques internes (axes du radar)
inner_metrics = sorted(df_norm['metric'].unique().tolist())
all_models_radar = sorted(df_norm['model'].unique().tolist())


def _make_radar(ax, labels, values, title=None, fill=True):
    # labels = noms de métriques ; values = scores 0..1 de même longueur
    N = len(labels)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    vals = list(values)
    # fermer le polygone
    angles += angles[:1]
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=2)
    if fill:
        ax.fill(angles, vals, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0.25", "0.5", "0.75", "1.0"], fontsize=8)
    ax.set_ylim(0, 1.0)
    if title:
        ax.set_title(title, pad=12, fontsize=12)


def plot_radar_for_model(model, fill=True, only_metrics=None):
    """
    Affiche un radar pour `model`:
      - Axes = toutes les métriques internes (ou un sous-ensemble via only_metrics)
      - Valeurs = score_norm (min-max par métrique, inversé)
    """
    d = df_norm[df_norm['model'] == model][['metric', 'score_norm']]

    # S'assure que l'ordre des axes est cohérent
    labels = inner_metrics if only_metrics is None else [m for m in inner_metrics if m in set(only_metrics)]
    # Valeurs (met 0.0 si la métrique manque pour ce modèle)
    val_by_metric = dict(zip(d['metric'], d['score_norm']))
    values = [val_by_metric.get(m, 0.0) for m in labels]

    fig = plt.figure(figsize=(7, 7))
    ax = plt.subplot(111, projection='polar')
    _make_radar(ax, labels, values, title=f"Qualité relative par métrique – {model}", fill=fill)
    plt.tight_layout()
    plt.show()


# show the widgets

widgets.VBox([widgets.Label("Radar des mean_abs_diff par modèle (normalisé par métrique)"), widgets.interactive(
    plot_radar_for_model,
    model=widgets.Dropdown(options=all_models_radar, value=all_models_radar[0], description="Modèle"),
    fill=widgets.Checkbox(value=True, description="Remplir"),
    only_metrics=widgets.SelectMultiple(options=inner_metrics, value=tuple(inner_metrics), description="Métriques",
                                        rows=min(10, len(inner_metrics)))
)])



In [ ]:
# %% [markdown]
# ## Radar superposé des mean_abs_diff (normalisé par métrrique) — plusieurs modèles sur un même graphe

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


# --- Normalisation min–max par métrique (inversée car mean_abs_diff = erreur) ---
def _minmax_inv(xmin, xmax, x):
    if not np.isfinite(xmin) or not np.isfinite(xmax) or np.isclose(xmin, xmax):
        return 1.0
    return float(1.0 - (x - xmin) / (xmax - xmin))


stats = (
    df_diff_by_model_metric
    .groupby('metric', as_index=False)['mean_abs_diff']
    .agg(metric_min='min', metric_max='max')
)

df_norm = df_diff_by_model_metric.merge(stats, on='metric', how='left').copy()
df_norm['score_norm'] = [
    _minmax_inv(r.metric_min, r.metric_max, r.mean_abs_diff) for r in df_norm.itertuples()
]

inner_metrics = sorted(df_norm['metric'].unique().tolist())
all_models_radar = sorted(df_norm['model'].unique().tolist())


# --- Radar util ---
def _radar_axes_labels(N):
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]
    return angles


def _plot_one_model_on_radar(ax, labels, values, label, color=None, marker=None, fill=False, alpha=0.25):
    # labels: axes names ; values: list in [0..1]
    N = len(labels)
    angles = _radar_axes_labels(N)
    vals = list(values) + [values[0]]
    ax.plot(angles, vals, linewidth=2, label=label, color=color, marker=marker)
    if fill:
        ax.fill(angles, vals, alpha=alpha, color=color)


def plot_radar_overlay(models_to_show, only_metrics=None, fill=False, show_markers=True):
    """
    Superpose plusieurs modèles sur un unique radar.
    - models_to_show: liste de modèles
    - only_metrics: sous-ensemble d'axes (sinon toutes les métriques internes)
    - fill: remplir les polygones
    """
    if not models_to_show:
        print("Sélectionne au moins un modèle.")
        return

    labels = inner_metrics if (only_metrics is None or len(only_metrics) == 0) else [m for m in inner_metrics if
                                                                                     m in set(only_metrics)]
    if len(labels) < 3:
        print("Il faut au moins 3 métriques pour un radar lisible.")
        return

    # Préparer couleurs/markers
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    markers = ['o', 's', 'D', '^', 'v', 'X', 'P', '*', 'h']  # au cas où > 7
    fig = plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, projection='polar')

    # Axe et ticks
    N = len(labels)
    angles = _radar_axes_labels(N)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0.25", "0.5", "0.75", "1.0"], fontsize=8)
    ax.set_ylim(0, 1.0)
    ax.set_title("Radar des mean_abs_diff (normalisés & inversés) — plus haut = meilleur", pad=12, fontsize=12)

    # Tracé, un polygone par modèle
    by_model = {m: df_norm[df_norm['model'] == m][['metric', 'score_norm']] for m in models_to_show}
    for i, m in enumerate(models_to_show):
        d = by_model[m]
        val_map = dict(zip(d['metric'], d['score_norm']))
        vals = [val_map.get(mm, 0.0) for mm in labels]
        col = colors[i % len(colors)]
        mrk = markers[i % len(markers)] if show_markers else None
        _plot_one_model_on_radar(ax, labels, vals, label=m, color=col, marker=mrk, fill=fill, alpha=0.18)

    # Légende propre, à l'extérieur
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0., title="Modèles", fontsize=9)
    plt.tight_layout()
    plt.show()


# --- Widgets ---
w_models = widgets.SelectMultiple(
    options=all_models_radar,
    value=tuple(all_models_radar),  # par défaut tous les modèles
    description="Modèles",
    rows=min(8, len(all_models_radar))
)
w_metrics = widgets.SelectMultiple(
    options=inner_metrics,
    value=tuple(inner_metrics),
    description="Métriques",
    rows=min(10, len(inner_metrics))
)
w_fill = widgets.Checkbox(value=False, description="Remplir les polygones")
w_mark = widgets.Checkbox(value=True, description="Afficher les marqueurs")

out = widgets.interactive_output(
    plot_radar_overlay,
    {
        'models_to_show': w_models,
        'only_metrics': w_metrics,
        'fill': w_fill,
        'show_markers': w_mark
    }
)

display(
    widgets.VBox([
        widgets.HTML("<b>Radar superposé des mean_abs_diff par métrique</b>"),
        widgets.HBox([w_models, w_metrics]),
        widgets.HBox([w_fill, w_mark]),
        out
    ])
)
